<a href="https://colab.research.google.com/github/Rollybuilds/Inventory_management_system.py/blob/main/Inventory_Management_System_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**PROJECT - 5 INVENTORY MANAGEMENT SYSTEM**

In [2]:
"""Inventory Management System """

import csv
from abc import ABC, abstractmethod

PRODUCTS_FILE = "products.csv"
REPORT_FILE = "inventory_report.txt"
TABLE_HEADER = (f"{'ID':<6}{'Name':<22}{'Category':<13}"
                f"{'Price':>10}{'Qty':>6}{'Value':>12}  Status")


# ---------------------------------------------------------------- Exception
class InventoryError(Exception):
    """Raised when an inventory action cannot be completed."""


# ------------------------------------------------- Abstract base class (abc)
class Product(ABC):
    """Base class for all products."""

    def __init__(self, product_id, name, price, quantity):
        self.__product_id = product_id    # private attributes (encapsulation)
        self.__name = ""
        self.__price = 0
        self.__quantity = 0
        self.name = name                  # uses the setters below for validation
        self.price = price
        self.quantity = quantity

    # Getters and setters
    @property
    def product_id(self):
        return self.__product_id

    @property
    def name(self):
        return self.__name

    @name.setter
    def name(self, value):
        if not value.strip():
            raise ValueError("Product name cannot be empty.")
        self.__name = value.strip()

    @property
    def price(self):
        return self.__price

    @price.setter
    def price(self, value):
        if value <= 0:
            raise ValueError("Price must be greater than zero.")
        self.__price = value

    @property
    def quantity(self):
        return self.__quantity

    @quantity.setter
    def quantity(self, value):
        if value < 0:
            raise ValueError("Quantity cannot be negative.")
        self.__quantity = value

    # Common actions
    def stock_value(self):
        return self.price * self.quantity

    def is_low_stock(self):
        return self.quantity <= self.low_stock_limit()

    def to_row(self):
        return [self.product_id, self.name, self.category(),
                f"{self.price:.2f}", self.quantity]

    def __str__(self):
        status = "LOW STOCK" if self.is_low_stock() else "OK"
        return (f"{self.product_id:<6}{self.name:<22}{self.category():<13}"
                f"{self.price:>10.2f}{self.quantity:>6}"
                f"{self.stock_value():>12.2f}  {status}")

    # Every child class must implement these
    @abstractmethod
    def category(self):
        """Return the category name of the product."""

    @abstractmethod
    def low_stock_limit(self):
        """Return the quantity at or below which stock is low."""


# ------------------------------------------------ Derived classes (inheritance)
class GroceryProduct(Product):
    def category(self):
        return "Grocery"

    def low_stock_limit(self):
        return 20


class ElectronicProduct(Product):
    def category(self):
        return "Electronic"

    def low_stock_limit(self):
        return 5


PRODUCT_TYPES = {"Grocery": GroceryProduct, "Electronic": ElectronicProduct}


# ---------------------------------------------------------------- Inventory
class Inventory:
    """Manages all products and saves them to a CSV file."""

    def __init__(self):
        self.__products = {}
        self.load_data()

    # ----- file handling
    def load_data(self):
        try:
            with open(PRODUCTS_FILE, newline="") as file:
                for row in csv.DictReader(file):
                    product_class = PRODUCT_TYPES[row["category"]]
                    product = product_class(int(row["id"]), row["name"],
                                            float(row["price"]), int(row["quantity"]))
                    self.__products[product.product_id] = product
        except FileNotFoundError:
            print("No saved products found. Starting with an empty inventory.")
        except (KeyError, ValueError):
            print("Some product records could not be read.")

    def save_data(self):
        try:
            with open(PRODUCTS_FILE, "w", newline="") as file:
                writer = csv.writer(file)
                writer.writerow(["id", "name", "category", "price", "quantity"])
                for product in self.__products.values():
                    writer.writerow(product.to_row())
        except OSError:
            print("Error: could not save products to file.")

    # ----- inventory actions
    def add_product(self, name, category, price, quantity):
        new_id = max(self.__products, default=100) + 1
        product = PRODUCT_TYPES[category](new_id, name, price, quantity)
        self.__products[new_id] = product
        self.save_data()
        return product

    def find_product(self, product_id):
        if product_id not in self.__products:
            raise InventoryError("No product found with that ID.")
        return self.__products[product_id]

    def update_product(self, product_id, price=None, quantity=None):
        """Change the price and/or quantity. None means 'keep as it is'."""
        product = self.find_product(product_id)
        if price is not None:
            product.price = price
        if quantity is not None:
            product.quantity = quantity
        self.save_data()
        return product

    def delete_product(self, product_id):
        product = self.find_product(product_id)
        del self.__products[product_id]
        self.save_data()
        return product

    def search(self, keyword):
        keyword = keyword.strip().lower()
        return [p for p in self.__products.values()
                if keyword in p.name.lower()
                or keyword in p.category().lower()
                or keyword == str(p.product_id)]

    def generate_report(self):
        """Build the inventory report and save it to a text file."""
        products = list(self.__products.values())
        low_stock = [p for p in products if p.is_low_stock()]
        total_units = sum(p.quantity for p in products)
        total_value = sum(p.stock_value() for p in products)

        line = "=" * len(TABLE_HEADER)
        lines = [line, "INVENTORY REPORT".center(len(TABLE_HEADER)), line,
                 TABLE_HEADER, "-" * len(TABLE_HEADER)]
        lines += [str(p) for p in products] or ["No products in inventory."]
        lines += ["-" * len(TABLE_HEADER),
                  f"Total products        : {len(products)}",
                  f"Total units in stock  : {total_units}",
                  f"Total inventory value : {total_value:.2f}",
                  f"Low stock products    : {len(low_stock)}"]
        for product in low_stock:
            lines.append(f"  - {product.name} (only {product.quantity} left)")
        lines.append(line)
        report = "\n".join(lines)

        try:
            with open(REPORT_FILE, "w") as file:
                file.write(report + "\n")
        except OSError:
            print("Error: could not save the report to file.")
        return report


# ---------------------------------------------------------------- Menu
def read_number(message, number_type=int):
    """Keep asking until the user enters a valid number."""
    while True:
        try:
            return number_type(input(message))
        except ValueError:
            print("Please enter a valid number.")


def read_optional_number(message, number_type):
    """Like read_number, but pressing Enter returns None (keep old value)."""
    while True:
        text = input(message).strip()
        if text == "":
            return None
        try:
            return number_type(text)
        except ValueError:
            print("Please enter a valid number or press Enter to skip.")


def show_products(products):
    if not products:
        print("No products found.")
        return
    print(TABLE_HEADER)
    for product in products:
        print(product)


def main():
    inventory = Inventory()

    menu = """
===== INVENTORY MANAGEMENT SYSTEM =====
1. Add product
2. Update product
3. Delete product
4. Search products
5. Generate inventory report
6. Exit
"""

    while True:
        print(menu)
        choice = input("Enter your choice (1-6): ").strip()

        try:
            if choice == "1":
                name = input("Product name: ")
                kind = input("Category (1-Grocery, 2-Electronic): ").strip()
                if kind not in ("1", "2"):
                    raise InventoryError("Invalid category.")
                category = "Grocery" if kind == "1" else "Electronic"
                price = read_number("Price: ", float)
                quantity = read_number("Quantity: ")
                product = inventory.add_product(name, category, price, quantity)
                print(f"Product added with ID {product.product_id}.")

            elif choice == "2":
                product = inventory.find_product(read_number("Product ID: "))
                show_products([product])
                print("Press Enter to keep the old value.")
                price = read_optional_number("New price: ", float)
                quantity = read_optional_number("New quantity: ", int)
                inventory.update_product(product.product_id, price, quantity)
                print("Product updated successfully.")

            elif choice == "3":
                product = inventory.find_product(read_number("Product ID: "))
                show_products([product])
                if input("Delete this product? (y/n): ").strip().lower() == "y":
                    inventory.delete_product(product.product_id)
                    print("Product deleted.")
                else:
                    print("Delete cancelled.")

            elif choice == "4":
                show_products(inventory.search(input("Search by name, category or ID: ")))

            elif choice == "5":
                print(inventory.generate_report())
                print(f"Report saved in {REPORT_FILE}.")

            elif choice == "6":
                inventory.save_data()
                print("Data saved. Goodbye!")
                break

            else:
                print("Invalid choice. Please select 1-6.")

        except (InventoryError, ValueError) as error:
            print(f"Error: {error}")


if __name__ == "__main__":
    main()


===== INVENTORY MANAGEMENT SYSTEM =====
1. Add product
2. Update product
3. Delete product
4. Search products
5. Generate inventory report
6. Exit

Enter your choice (1-6): 1
Product name: Laptop
Category (1-Grocery, 2-Electronic): 2
Price: 100000
Quantity: 5
Product added with ID 104.

===== INVENTORY MANAGEMENT SYSTEM =====
1. Add product
2. Update product
3. Delete product
4. Search products
5. Generate inventory report
6. Exit

Enter your choice (1-6): 1
Product name: Powerbank
Category (1-Grocery, 2-Electronic): 2
Price: 800
Quantity: 70
Product added with ID 105.

===== INVENTORY MANAGEMENT SYSTEM =====
1. Add product
2. Update product
3. Delete product
4. Search products
5. Generate inventory report
6. Exit

Enter your choice (1-6): 1
Product name: Upma
Category (1-Grocery, 2-Electronic): 1
Price: 60
Quantity: 60
Product added with ID 106.

===== INVENTORY MANAGEMENT SYSTEM =====
1. Add product
2. Update product
3. Delete product
4. Search products
5. Generate inventory report
